In [3]:
import os
import xarray as xr
import pandas as pd
import glob
import xarray as xr

import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature


PATH = "/Users/lb962/Documents/GitHub/ESL/data/ml_ready4/*.nc"

# Get sorted list of files
files = sorted(glob.glob(PATH))

# Open each file into a list of datasets
datasets = [xr.open_dataset(f) for f in files]

# Example: inspect or process one
for i, ds in enumerate(datasets):
    print(f"File {i}: {files[i]}")
    print(ds)

# If you want them in memory:
datasets = [ds.load() for ds in datasets]

def preprocess(ds):
    return ds.sortby("station") 

datasets = [preprocess(xr.open_dataset(f)) for f in files]
ds = xr.concat(datasets, dim="station")


File 0: /Users/lb962/Documents/GitHub/ESL/data/ml_ready4/small_ds010.nc
<xarray.Dataset>
Dimensions:            (station: 10, valid_time: 750192)
Coordinates:
  * station            (station) int64 1 2 3 4 5 6 7 8 9 10
  * valid_time         (valid_time) datetime64[ns] 1940-01-01 ... 2025-07-30T...
    latitude           float64 ...
    longitude          float64 ...
    station_latitude   (station) float64 ...
    station_longitude  (station) float64 ...
Data variables: (12/14)
    u10                (station, valid_time) float32 ...
    v10                (station, valid_time) float32 ...
    d2m                (station, valid_time) float32 ...
    t2m                (station, valid_time) float32 ...
    msl                (station, valid_time) float32 ...
    sst                (station, valid_time) float32 ...
    ...                 ...
    mwd                (station, valid_time) float32 ...
    mwp                (station, valid_time) float32 ...
    swh                (station,

In [4]:
from numba import njit
import numpy as np
import xarray as xr
from numba import njit

@njit(cache=True, fastmath=True)
def _mask_1d(values, k_eff, mean, std, safe_lo, safe_hi,
             hard_clip_eff, p95, p95_margin_eff,
             q1, q3, use_quantiles, iqr_mult):
    n = values.shape[0]
    dev = np.zeros(n, dtype=np.bool_)
    last_kept = np.nan; have_last=False; prev_was_kept=False
    z_enabled = np.isfinite(std) and std > 0.0
    p95_gate = p95 + p95_margin_eff if np.isfinite(p95) else np.inf
    iqr = q3 - q1
    lo_iqr = q1 - iqr_mult * iqr
    hi_iqr = q3 + iqr_mult * iqr
    for i in range(n):
        x = values[i]
        if not np.isfinite(x): dev[i]=True; prev_was_kept=False; continue
        if np.abs(x) > hard_clip_eff: dev[i]=True; prev_was_kept=False; continue
        if (x >= safe_lo) and (x <= safe_hi): dev[i]=False; last_kept=x; have_last=True; prev_was_kept=True; continue
        if use_quantiles and ((x < lo_iqr) or (x > hi_iqr)): dev[i]=True; prev_was_kept=False; continue
        if z_enabled:
            z = (x - mean) / std
            if np.abs(z) > k_eff: dev[i]=True; prev_was_kept=False; continue
        if not prev_was_kept:
            if x <= p95_gate: dev[i]=False; last_kept=x; have_last=True; prev_was_kept=True
            else: dev[i]=True; prev_was_kept=False
            continue
        if have_last:
            allowed = 1.5 if (-1.0 <= last_kept <= 1.0) else 0.5
            if np.abs(x - last_kept) > allowed: dev[i]=True; prev_was_kept=False; continue
        dev[i]=False; last_kept=x; have_last=True; prev_was_kept=True
    return dev

# FAST block-drift mask: EW-quantile (median) + run-length
@njit(cache=True, fastmath=True)
def _block_mask_1d(values, med_thresh, min_run, alpha):
    n = values.shape[0]
    block = np.zeros(n, dtype=np.bool_)
    m = 0.0                        # EW median estimate (starts near 0 for residuals)
    run = 0                        # consecutive samples with |m| > threshold
    for i in range(n):
        x = values[i]
        if np.isfinite(x):
            # Robbins–Monro quantile update (tau=0.5 -> median)
            m += alpha * (0.5 - (1.0 if x < m else 0.0))
            if np.abs(m) > med_thresh:
                run += 1
                if run == min_run:
                    # backfill the initial block once threshold run is reached
                    start = i - min_run + 1
                    for j in range(start, i + 1):
                        block[j] = True
                elif run > min_run:
                    block[i] = True
            else:
                run = 0
        else:
            run = 0  # NaN breaks the block
    return block
@njit(cache=True, fastmath=True)
def _yrmean_block_mask_1d(values, yr_mean_vals, good_year_vals,
                          mean_abs_thresh, band, min_run):
    """
    Flag contiguous runs that stay near the (shifted) yearly mean, but only
    in years where |yearly mean| > mean_abs_thresh and coverage was 'good'.
    """
    n = values.shape[0]
    block = np.zeros(n, dtype=np.bool_)
    run = 0
    start = 0
    for i in range(n):
        x = values[i]
        if not np.isfinite(x):
            run = 0
            continue

        if not good_year_vals[i]:
            # year not eligible: reset run
            run = 0
            continue

        ymean = yr_mean_vals[i]
        if not np.isfinite(ymean) or (np.abs(ymean) <= mean_abs_thresh):
            # yearly mean not "shifted" enough
            run = 0
            continue

        # Are we "near" the shifted yearly mean?
        if np.abs(x - ymean) <= band:
            if run == 0:
                start = i
            run += 1
            if run == min_run:
                for j in range(start, i + 1):
                    block[j] = True
            elif run > min_run:
                block[i] = True
        else:
            run = 0
    return block
def filter_residual(
    ds: xr.Dataset,
    var: str = "residual",
    dim: str | None = None,
    base_k: float = 5.0,
    safe_keep_band: tuple[float, float] = (-2.0, 2.0),
    hard_clip: float = 3.0,
    p95_margin: float = 0.5,
    station_dim: str = "station",
    min_frac_year: float = 0.5,
    kurt_hard: float = 1.0,
    # mean-shift block knobs
    year_mean_abs_thresh: float = 1.0,   # consider a year "shifted" if |year mean| > 1 m
    block_band: float = 0.5,             # samples within ±band of the yearly mean count toward a block
    min_run: int = 60,                   # consecutive samples near the mean to declare a block
):
    da = ds[var]; dim = da.dims[0] if dim is None else dim
    ds = ds.sortby(dim); da = ds[var].astype("float64")

    # --- robust stats (unchanged)
    mu   = da.mean(dim=dim, skipna=True)
    sig  = da.std(dim=dim,  skipna=True)
    p95  = da.quantile(0.95, dim=dim, skipna=True)
    qs   = da.quantile([0.25, 0.75], dim=dim, skipna=True)
    q1, q3 = qs.sel(quantile=0.25), qs.sel(quantile=0.75)

    m2 = da.var(dim=dim, skipna=True)
    m4 = ((da - mu) ** 4).mean(dim=dim, skipna=True)
    ex_kurt = xr.where(m2 > 0, m4 / (m2 ** 2) - 3.0, 0.0)
    heavy = ex_kurt > kurt_hard

    k_eff          = xr.where(heavy, base_k / 2.5, base_k)
    iqr_mult       = xr.where(heavy, 1.0, 1.5)
    hard_clip_eff  = xr.where(heavy, 2.0, hard_clip)
    p95_margin_eff = xr.where(heavy, 0.1, p95_margin)
    use_quantiles  = heavy

    dev_point = xr.apply_ufunc(
        _mask_1d,
        da, k_eff, mu, sig,
        xr.DataArray(safe_keep_band[0]), xr.DataArray(safe_keep_band[1]),
        hard_clip_eff, p95, p95_margin_eff,
        q1, q3, use_quantiles, iqr_mult,
        input_core_dims=[[dim], [], [], [], [], [], [], [], [], [], [], [], []],
        output_core_dims=[[dim]],
        vectorize=True, dask="parallelized",
        output_dtypes=[bool],
    )

    # --- yearly mean and coverage (unchanged)
    present = da.notnull()
    cov_year = present.groupby(f"{dim}.year").mean(dim=dim)         # fraction present per year
    yr_mean  = da.groupby(f"{dim}.year").mean(dim=dim, skipna=True)  # mean per year
    good_year = cov_year > min_frac_year

    # --- NEW: broadcast per-year values back to each timestamp by selecting with a per-sample 'year' indexer
    # Ensure time axis is datetime64
    if not np.issubdtype(ds[dim].dtype, np.datetime64):
        # If your coord isn't datetime64, convert it here (one-time)
        ds = ds.assign_coords({dim: xr.converters.to_datetime64(ds[dim])})
        da = ds[var].astype("float64")

    year_indexer = xr.DataArray(
        ds[dim].dt.year,         # vector of years for every sample along `dim`
        dims=dim,
        coords={dim: ds[dim]}
    )

    # These now have the SAME dims as `da` (broadcasted back from 'year')
    yr_mean_full   = yr_mean.sel(year=year_indexer)
    good_year_full = good_year.sel(year=year_indexer)

    # --- block detector: runs near the shifted yearly mean
    dev_block = xr.apply_ufunc(
        _yrmean_block_mask_1d,
        da,
        yr_mean_full,
        good_year_full,
        xr.DataArray(year_mean_abs_thresh),
        xr.DataArray(block_band),
        xr.DataArray(np.int64(min_run)),
        input_core_dims=[[dim], [dim], [dim], [], [], []],
        output_core_dims=[[dim]],
        vectorize=True, dask="parallelized",
        output_dtypes=[bool],
    )

    deviant = dev_point | dev_block

    out = ds.copy()
    out[f"{var}_is_deviant"] = deviant
    out[var] = da.where(~deviant)

    if station_dim in out.dims:
        extra = [d for d in cov_year.dims if d not in (station_dim, "year")]
        cov_check = cov_year if not extra else cov_year.mean(dim=extra)
        keep_station = (cov_check > min_frac_year).any(dim="year")
        out = out.isel({station_dim: keep_station})

    return out, out[f"{var}_is_deviant"]
# ds: xr.Dataset with a variable named "residual" along dimension "time"
#ds_filtered, dev_mask = filter_residual(ds, var="residual", dim="valid_time")
#ds = ds_filtered

In [6]:
import os
import numpy as np
import xarray as xr

# --- config ---
STATION_DIM = "station"
TIME_DIM = "valid_time"
VAR_NAME = "residual"
OUT_DIR = "/Users/lb962/Documents/GitHub/ESL/data/ml4_removed"

os.makedirs(OUT_DIR, exist_ok=True)

def _station_label(ds_station, i, station_dim=STATION_DIM):
    """
    Choose a filename label:
    - prefer a 'station_id' or 'id' coordinate/variable if present
    - else use the station coordinate value if it's numeric/int-like
    - else fall back to the zero-padded index
    """
    # Try common id fields first
    for key in ("station_id", "id", "wmo_id", "name"):
        if key in ds_station.coords:
            val = ds_station.coords[key].values
            if np.size(val) == 1:
                return str(np.array(val).item())
        if key in ds_station:
            val = ds_station[key].values
            if np.size(val) == 1:
                return str(np.array(val).item())
    # Try the station coordinate itself
    if station_dim in ds_station.coords:
        val = ds_station[station_dim].values
        if np.size(val) == 1:
            try:
                # stringify but keep simple ints clean
                return str(np.array(val).item())
            except Exception:
                pass
    # Fallback: index
    return f"{i:05d}"

# Optional: compression settings for smaller files
def _nc_encoding(ds_single, var_name=VAR_NAME):
    enc = {}
    if var_name in ds_single:
        enc[var_name] = {"zlib": True, "complevel": 4, "dtype": "float32"}
    dev_name = f"{var_name}_is_deviant"
    if dev_name in ds_single:
        enc[dev_name] = {"zlib": True, "complevel": 4, "dtype": "int8"}
    # compress coords lightly
    for c in ds_single.coords:
        if np.issubdtype(ds_single[c].dtype, np.number):
            enc[c] = {"zlib": True, "complevel": 1}
    return enc

n_stations = ds.sizes[STATION_DIM]
print(f"Processing {n_stations} stations...")

for i in range(n_stations):
    ds_i = ds.isel({STATION_DIM: i})

    # Ensure time is datetime64 (your filter handles this, but doing it once per station is cheap)
    if not np.issubdtype(ds_i[TIME_DIM].dtype, np.datetime64):
        ds_i = ds_i.assign_coords({TIME_DIM: xr.converters.to_datetime64(ds_i[TIME_DIM])})

    # Apply your exact outlier filter per station
    ds_filt_i, dev_mask_i = filter_residual(
        ds_i,
        var=VAR_NAME,
        dim=TIME_DIM,
        station_dim=STATION_DIM,   # keep the 1-length station dim in the file
        # (all your default knobs are used; override here if you want)
    )

    # Build filename
    label = _station_label(ds_i, i, station_dim=STATION_DIM)
    safe_label = "".join(ch if ch.isalnum() or ch in ("_", "-", ".") else "_" for ch in str(label))
    out_path = os.path.join(OUT_DIR, f"station_{safe_label}.nc")

    # Save a compact file
    enc = _nc_encoding(ds_filt_i, var_name=VAR_NAME)
    ds_filt_i.to_netcdf(out_path, mode="w", format="NETCDF4", encoding=enc)

    print(f"[{i+1}/{n_stations}] wrote {out_path}")


Processing 295 stations...
[1/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_1.nc
[2/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_2.nc
[3/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_3.nc
[4/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_4.nc
[5/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_5.nc
[6/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_6.nc
[7/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_7.nc
[8/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_8.nc
[9/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_9.nc
[10/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_10.nc
[11/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_161.nc
[12/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_162.nc
[13/295] wrote /Users/lb962/D

/Users/lb962/miniconda3/envs/ESL/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1545: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(


[73/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_276.nc
[74/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_277.nc
[75/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_326.nc
[76/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_327.nc
[77/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_328.nc
[78/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_329.nc
[79/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_330.nc
[80/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_331.nc
[81/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_332.nc
[82/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_333.nc
[83/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_334.nc
[84/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_335.nc
[85/295] wrote /Users/lb962/

/Users/lb962/miniconda3/envs/ESL/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1545: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(


[142/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_466.nc
[143/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_467.nc
[144/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_468.nc
[145/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_469.nc
[146/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_470.nc
[147/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_471.nc


...

[266/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_113.nc
[267/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_114.nc
[268/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_115.nc
[269/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_120.nc
[270/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_121.nc
[271/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_122.nc
[272/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_123.nc
[273/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_124.nc
[274/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_125.nc
[275/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_126.nc
[276/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_127.nc
[277/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_128.nc
[278/295] wrote 

/Users/lb962/miniconda3/envs/ESL/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1545: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(


[283/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_145.nc
[284/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_146.nc
[285/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_147.nc
[286/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_148.nc
[287/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_151.nc
[288/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_152.nc
[289/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_153.nc
[290/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_154.nc
[291/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_156.nc
[292/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_157.nc
[293/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_158.nc


/Users/lb962/miniconda3/envs/ESL/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1545: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(


[294/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_159.nc
[295/295] wrote /Users/lb962/Documents/GitHub/ESL/data/ml4_removed/station_160.nc
